# Challenge 2: Optimization of Battery Usage in the Installation

In this notebook, we address Objective 2 of the Repsol IE Sustainability Challenge:

- **Objective:** Optimize the use of a theoretical battery (100 kWh capacity, 100 kW max charge/discharge, one charge/discharge cycle per day) to maximize self‑consumption of solar energy and reduce grid dependence.

We will:

1. Load and prepare data (solar generation predictions, actual photovoltaic consumption, and grid consumption).
2. Ensure proper datetime handling (including timezone conversion) and filter for September 2024.
3. Merge datasets and compute the surplus solar energy.
4. Simulate battery operation using an improved strategy.
5. Compute the Self‑Consumption Ratio (Ra) as a business metric.
6. Export the final simulation results to CSV for submission.

Let's begin!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import pytz

print('Libraries imported successfully!')

## 1) Data Loading & Preparation

We load three datasets:

- **Solar Generation Predictions:** (from `september_predictions3.csv`)
- **Actual Photovoltaic Consumption:** (from `Consumo_Fotovoltaica.csv`; date column is "FECHA")
- **Grid Consumption:** (from `Consumo.csv`; date column is "FECHA")

We convert the datetime columns to the local time zone (`Europe/Madrid`) and filter for September 2024.

In [ ]:
# Load solar generation predictions; assume the column is already 'datetime'
df_gen = pd.read_csv('september_predictions3.csv', parse_dates=['datetime'])

# Load actual photovoltaic consumption; the date column is 'FECHA'
df_consumption = pd.read_csv('Consumo_Fotovoltaica.csv', parse_dates=['FECHA'])
df_consumption.rename(columns={'FECHA': 'datetime'}, inplace=True)

# Load grid consumption data; the date column is 'FECHA'
df_grid = pd.read_csv('Consumo.csv', parse_dates=['FECHA'])
df_grid.rename(columns={'FECHA': 'datetime'}, inplace=True)

# Helper function to ensure datetime is in the desired timezone
def ensure_local_timezone(df, col, tz='Europe/Madrid'):
    if df[col].dt.tz is None:
        df[col] = df[col].dt.tz_localize('UTC')
    df[col] = df[col].dt.tz_convert(tz)
    return df

local_tz = 'Europe/Madrid'
df_gen = ensure_local_timezone(df_gen, 'datetime', local_tz)
df_consumption = ensure_local_timezone(df_consumption, 'datetime', local_tz)
df_grid = ensure_local_timezone(df_grid, 'datetime', local_tz)

# Filter for September 2024
sep_start = pd.Timestamp('2024-09-01 00:00:00', tz=local_tz)
sep_end   = pd.Timestamp('2024-09-30 23:00:00', tz=local_tz)

df_sep_gen = df_gen[(df_gen['datetime'] >= sep_start) & (df_gen['datetime'] <= sep_end)].copy()
df_sep_cons = df_consumption[(df_consumption['datetime'] >= sep_start) & (df_consumption['datetime'] <= sep_end)].copy()
df_sep_grid = df_grid[(df_grid['datetime'] >= sep_start) & (df_grid['datetime'] <= sep_end)].copy()

print('Solar generation predictions shape:', df_sep_gen.shape)
print('Photovoltaic consumption shape:', df_sep_cons.shape)
print('Grid consumption shape:', df_sep_grid.shape)
print('Expected rows:', 30 * 24)

## 2) Merge & Compute Surplus Energy

We merge the solar generation predictions with the actual consumption data on `datetime` and calculate the surplus energy as:

```
Surplus = Predicted Solar Generation - Actual Photovoltaic Consumption
```

Negative values are clipped to 0.

In [ ]:
# Merge solar generation predictions with consumption
# Note: We assume df_sep_gen has a column 'pv_generation_pred' for predictions
df_sep = pd.merge(df_sep_gen[['datetime', 'pv_generation_pred']], 
                  df_sep_cons[['datetime', 'TOTAL_KWH_ENERGIA']], 
                  on='datetime', 
                  how='left')

# Calculate surplus energy
df_sep['excess_energy'] = df_sep['pv_generation_pred'] - df_sep['TOTAL_KWH_ENERGIA']
df_sep['excess_energy'] = df_sep['excess_energy'].clip(lower=0)

print('Surplus energy calculated. Sample:')
display(df_sep.head(5))

## 3) Improved Battery Simulation

We simulate a theoretical battery for each day in September with these rules:

- **Capacity:** 100 kWh
- **Max Charge/Discharge Power:** 100 kW per hour
- **Charging:** Battery charges with available surplus energy until full
- **Discharging:** Battery discharges at the hour with the highest grid consumption

This simulation will produce new columns:

- `battery_charge`: Battery level after each hour
- `energy_charged`: Energy charged during each hour
- `energy_discharged`: Energy discharged at the chosen hour

We then add a column `energy_recovered` (energy discharged) for business metric calculations.

In [ ]:
def simulate_battery(df_day, capacity=100, max_power=100):
    """
    Simulate battery operation for one day.
    
    Parameters:
        df_day: DataFrame for one day with 'excess_energy' and 'TOTAL_KWH_ENERGIA'.
        capacity: Battery capacity in kWh.
        max_power: Maximum charge/discharge in kWh per hour.
    
    Returns:
        DataFrame with additional columns:
            - 'battery_charge'
            - 'energy_charged'
            - 'energy_discharged'
    """
    battery_level = 0
    charge_list = []
    energy_charged = []
    energy_discharged = np.zeros(len(df_day))
    
    # Charge battery based on available surplus each hour
    for i, row in df_day.iterrows():
        available_surplus = row['excess_energy']
        charge = min(available_surplus, max_power, capacity - battery_level)
        battery_level += charge
        charge_list.append(battery_level)
        energy_charged.append(charge)
    
    df_day = df_day.copy()
    df_day['battery_charge'] = charge_list
    df_day['energy_charged'] = energy_charged
    
    # Determine discharge hour: choose hour with highest grid consumption
    if 'TOTAL_KWH_ENERGIA' in df_day.columns:
        discharge_idx = df_day['TOTAL_KWH_ENERGIA'].idxmax()
    else:
        discharge_idx = df_day.index[-1]
    
    # Discharge all energy (subject to max power constraint) at that hour
    discharge_amount = min(battery_level, max_power)
    energy_discharged[df_day.index.get_loc(discharge_idx)] = discharge_amount
    battery_level -= discharge_amount
    
    df_day['energy_discharged'] = energy_discharged
    df_day['energy_recovered'] = df_day['energy_discharged']
    
    return df_day

# Add a 'date' column and apply simulation for each day
df_sep['date'] = df_sep['datetime'].dt.date
df_simulated = df_sep.groupby('date').apply(lambda d: simulate_battery(d.copy()))

print('Battery simulation completed.')
display(df_simulated.head(10))

## 4) Compute Business Metrics

### Self‑Consumption Ratio (Ra)

We define Ra as:

```
Ra = (Direct Solar Consumption + Energy Recovered from Battery) / Total Solar Generation
```

Here, we assume direct consumption is given by `TOTAL_KWH_ENERGIA` and energy recovered by battery is `energy_discharged`.

We calculate Ra for September 2024.

In [ ]:
# Compute solar used: actual consumption plus energy discharged from battery
df_simulated['solar_used'] = df_simulated['TOTAL_KWH_ENERGIA'] + df_simulated['energy_discharged']

total_solar_gen = df_simulated['pv_generation_pred'].sum()
total_solar_used = df_simulated['solar_used'].sum()

Ra = total_solar_used / total_solar_gen
print(f"Self-Consumption Ratio (Ra): {Ra:.4f}")

## 5) Export Final Predictions

We ensure that our final simulation for September has exactly 720 rows and is sorted by datetime.
We then export the following columns to CSV:

- datetime
- pv_generation_pred
- TOTAL_KWH_ENERGIA
- energy_discharged
- solar_used

In [ ]:
# Sort the simulated data by datetime
df_simulated.sort_values('datetime', inplace=True)

# Verify row count
print('Number of rows in simulated September data:', df_simulated.shape[0])

# Export final predictions to CSV
export_cols = ['datetime', 'pv_generation_pred', 'TOTAL_KWH_ENERGIA', 'energy_discharged', 'solar_used']
df_simulated[export_cols].to_csv('september_predictions_battery_simulation.csv', index=False)
print("Predictions exported to 'september_predictions_battery_simulation.csv'")

## Wrap-Up & Conclusion

In this notebook we:

1. Loaded solar generation predictions, photovoltaic consumption, and grid consumption data.
2. Converted datetime columns to local time and filtered for September 2024 (ensuring 720 rows).
3. Merged the datasets and calculated surplus solar energy.
4. Simulated battery operation with an improved strategy (charging with surplus and discharging at peak grid consumption).
5. Computed the Self‑Consumption Ratio (Ra) as a key business metric.
6. Exported the final simulation results for further evaluation.

Further refinements could include more dynamic battery simulation, incorporation of carbon intensity data, and advanced feature engineering. Iterative improvements will help achieve a lower MAE and better business metrics.

Good luck and keep iterating!